In [1]:
import pandas as pd

## Sensors

In [2]:
min_list=[]
max_list=[]
for i in range(1, 36+1):
    dfSensor = pd.read_excel('../data/DraginoSoilMositure_Morocco_Season1.xlsx', sheet_name=f'Sensor {i}',
                             usecols=[0, 1])
    dfSensor.rename(columns={"Row Labels": "datetime", "Average of water_SOIL": "sm_value"}, inplace=True)
    dfSensor["datetime"] = pd.to_datetime(dfSensor["datetime"])
    min_list.append(dfSensor["datetime"].min())
    max_list.append(dfSensor["datetime"].max())
print(f"Min: {min_list}")
print(f"Max: {max_list}")
max_abs=min(max_list)
min_abs=max(min_list)
print(min_abs)
print(max_abs)
full_index = pd.date_range(min_abs, max_abs, freq="h")

Min: [Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestam

In [3]:
sensors = {}
for i in range(1, 36+1):
    dfSensor = pd.read_excel('../data/DraginoSoilMositure_Morocco_Season1.xlsx', sheet_name=f'Sensor {i}',
                             usecols=[0, 1])
    dfSensor.rename(columns={"Row Labels": "datetime", "Average of water_SOIL": "sm_value"}, inplace=True)

    dfSensor["datetime"] = pd.to_datetime(dfSensor["datetime"])
    dfSensor.set_index("datetime", inplace=True)
    dfSensor = dfSensor.reindex(full_index)
    dfSensor["sm_value"] = dfSensor["sm_value"].interpolate()
    sensors[i] = dfSensor


## Meteo

In [4]:
dfMeteo = pd.read_csv('../data/open-meteo.csv')
dfMeteo["datetime"] = pd.to_datetime(dfMeteo["datetime"])

dfMeteo.set_index("datetime", inplace=True)
dfMeteo = dfMeteo.reindex(full_index)
print(dfMeteo.index.min())
print(dfMeteo.index.max())
print(dfMeteo.shape)

2024-12-12 12:00:00
2025-05-15 12:00:00
(3697, 11)


In [5]:
dfMeteo_F = pd.read_csv('../data/open-meteo-Forecast.csv')
dfMeteo_F["datetime"] = pd.to_datetime(dfMeteo_F["datetime"])

dfMeteo_F.set_index("datetime", inplace=True)
dfMeteo_F = dfMeteo_F.reindex(full_index)
print(dfMeteo_F.index.min())
print(dfMeteo_F.index.max())
print(dfMeteo_F.shape)

2024-12-12 12:00:00
2025-05-15 12:00:00
(3697, 9)


In [6]:
dfMeteoStation = pd.read_excel('../data/WeatherStation_Slimania_Season1.xlsx', usecols=[1,2,3,4,5,6,7,8,9])
dfMeteoStation["datetime"] = dfMeteoStation["datetime"].dt.round('h')
dfMeteoStation_rev = dfMeteoStation.groupby('datetime').mean()

dfMeteoStation_rev = dfMeteoStation_rev.reindex(full_index)

## LAI

In [33]:
dfLAI = pd.read_excel('../data/LAI_data_S1.xlsx', sheet_name=f'Sheet1')
dfLAI["datetime"] = pd.to_datetime(dfLAI["datetime"]) + pd.to_timedelta("12:00:00")
dfLAI = dfLAI.set_index("datetime")
dfLAI = dfLAI.reindex(full_index)

values = {
    "LAI1-CROPWAT-Tititcaca": 0,
    "LAI2-Sensor-Tiititcaca": 0,
    "LAI3-Farmer-Titicaca": 0,
    "LAI4-CROPWAT-ICBA": 0,
    "LAI5-Sensor-ICBA": 0,
    "LAI6-Farmer-ICBA": 0
}
dfLAI.loc["2024-12-12 12:00:00"] = values


In [35]:
dfLAI[['LAI1-CROPWAT-Tititcaca', 'LAI2-Sensor-Tiititcaca', 'LAI3-Farmer-Titicaca', 'LAI4-CROPWAT-ICBA', 'LAI5-Sensor-ICBA', 'LAI6-Farmer-ICBA']] = dfLAI[['LAI1-CROPWAT-Tititcaca', 'LAI2-Sensor-Tiititcaca', 'LAI3-Farmer-Titicaca', 'LAI4-CROPWAT-ICBA', 'LAI5-Sensor-ICBA', 'LAI6-Farmer-ICBA']].interpolate()

In [36]:

mapToSensor = {'LAI1-CROPWAT-Tititcaca': [3, 12, 16],
               'LAI2-Sensor-Tiititcaca': [2, 10, 18],
               'LAI3-Farmer-Titicaca': [5, 7, 15],
               'LAI4-CROPWAT-ICBA': [1, 8, 17],
               'LAI5-Sensor-ICBA': [4, 11, 13],
               'LAI6-Farmer-ICBA': [6, 9, 14]}

for i in range(1, 18+1):
    for k, v in mapToSensor.items():
        if i in v:
            dfLAI[f'f{i}'] = dfLAI[k]

dfLAI.drop(['LAI1-CROPWAT-Tititcaca', 'LAI2-Sensor-Tiititcaca', 'LAI3-Farmer-Titicaca', 'LAI4-CROPWAT-ICBA', 'LAI5-Sensor-ICBA', 'LAI6-Farmer-ICBA'], axis=1, inplace=True)


In [37]:
dfLAI.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18
2024-12-12 12:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2024-12-12 13:00:00,0.000541,0.000088,0.000475,0.000270,0.000183,0.000154,0.000183,0.000541,0.000154,0.000088,0.000270,0.000475,0.000270,0.000154,0.000183,0.000475,0.000541,0.000088
2024-12-12 14:00:00,0.001082,0.000175,0.000950,0.000541,0.000365,0.000307,0.000365,0.001082,0.000307,0.000175,0.000541,0.000950,0.000541,0.000307,0.000365,0.000950,0.001082,0.000175
2024-12-12 15:00:00,0.001623,0.000263,0.001425,0.000811,0.000548,0.000461,0.000548,0.001623,0.000461,0.000263,0.000811,0.001425,0.000811,0.000461,0.000548,0.001425,0.001623,0.000263
2024-12-12 16:00:00,0.002164,0.000351,0.001901,0.001082,0.000731,0.000614,0.000731,0.002164,0.000614,0.000351,0.001082,0.001901,0.001082,0.000614,0.000731,0.001901,0.002164,0.000351


# Irrigation

In [38]:
dfIRR=0

In [39]:
dfIRR = pd.DataFrame(columns=['f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11','f12','f13','f14','f15','f16','f17','f18'], index=full_index)
dfIRR.fillna(0.0, inplace=True)

/var/folders/66/rpqcrym93h90yphhq7kgwhb00000gn/T/ipykernel_31168/450902979.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfIRR.fillna(0.0, inplace=True)


In [40]:
dfIRR.columns

Index(['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11',
       'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18'],
      dtype='object')

In [41]:
dfIrrigation = pd.read_excel('../data/IrrigationEvents_Morocco_Season1.xlsx', sheet_name='Irrigation')
dfIrrigation.rename(columns={"Date": "datetime", "Irrigation Duration": "duration", "Irrigation Volume": "volume"},
                     inplace=True)
dfIrrigation.drop(['Soil Mositure before irrigation (only Sensor)','duration'], axis=1, inplace=True)
dfIrrigation["datetime"] = pd.to_datetime(dfIrrigation["datetime"]) + pd.to_timedelta("12:00:00")
dfIrrigation.sort_values("datetime", ascending=True, inplace=True)
CW = ['f1', 'f3', 'f8', 'f12', 'f16','f17']
F = ['f5', 'f6', 'f7', 'f9', 'f14', 'f15']
#S = ['f2', 'f4', 'f10', 'f11', 'f13','f18']
dfIrrigation["Plot"] = dfIrrigation["Plot"].apply(
    lambda x: CW if x == "CROPWAT" else F if x == "Farmer" else [f"f{x}"]
)


In [42]:
for irr_ev in dfIrrigation.iterrows():
    #print(irr_ev[1]['Plot'])
    for p in irr_ev[1]['Plot']:
        dfIRR.loc[pd.to_datetime(irr_ev[1]['datetime']), p] = irr_ev[1]['volume']
        #print([pd.to_datetime(irr_ev[1]['datetime']), p])
        #print(dfIRR.loc[irr_ev[1]['datetime'], p])
        #pd.to_datetime(irr_ev[1]['datetime']),

In [30]:
print(sensors[1].shape, sensors[1].columns)
print(dfIRR.shape, dfIRR.columns)
print(dfMeteo.shape, dfMeteo.columns)

(3697, 1) Index(['sm_value'], dtype='object')
(3697, 18) Index(['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11',
       'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18'],
      dtype='object')
(3697, 11) Index(['temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration'],
      dtype='object')


## Save dataset

In [44]:
meteo_fields = ['temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration']
meteo_fields_forecast = ['temperature_2m_f','relative_humidity_2m_f','cloud_cover_f','wind_speed_10m_f','wind_direction_10m_f','soil_temperature_0cm_f','soil_temperature_6cm_f','soil_temperature_18cm_f','rain_f']

meteo_station_fields = ['Temp', 'Hum', 'Int', 'UVI', 'WS', 'WD', 'RG', 'BP']
all_fields = []

for i, field in enumerate(['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11',
       'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18'], start=1):
    df_new = pd.DataFrame(columns=['s_b','s_w','LAI','irr','temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration'], index=full_index)
    df_new['irr'] = dfIRR[field]
    df_new['s_b'] = sensors[i*2]['sm_value']
    df_new['s_w'] = sensors[i*2-1]['sm_value']
    df_new['LAI'] = dfLAI[f'{field}']
    df_new[meteo_fields] = dfMeteo[meteo_fields]
    df_new[meteo_station_fields] = dfMeteoStation_rev[meteo_station_fields]
    df_new[meteo_fields_forecast] = dfMeteo_F[meteo_fields_forecast]
    df_new.to_csv(f'fields/field_{field}',index_label='datetime')


In [ ]:
df = pd.read_csv('fieldsComplete/field_f1')
columns = df.columns[1:-1]
for i, feat in enumerate(columns):
    print(f'{i} - {feat}, ')

0 - s_a, 
1 - s_b, 
2 - irr, 
3 - temperature_2m, 
4 - relative_humidity_2m, 
5 - cloud_cover, 
6 - wind_speed_10m, 
7 - wind_direction_100m, 
8 - soil_temperature_0_to_7cm, 
9 - soil_temperature_7_to_28cm, 
10 - soil_temperature_28_to_100cm, 
11 - rain, 
12 - precipitation, 
13 - evapotranspiration, 
14 - LAI, 
15 - Temp, 
16 - Hum, 
17 - Int, 
18 - UVI, 
19 - WS, 
20 - WD, 
21 - RG, 
22 - BP, 
23 - temperature_2m_f, 
24 - relative_humidity_2m_f, 
25 - cloud_cover_f, 
26 - wind_speed_10m_f, 
27 - wind_direction_10m_f, 
28 - soil_temperature_0cm_f, 
29 - soil_temperature_6cm_f, 
30 - soil_temperature_18cm_f, 


In [ ]:
import plotly.graph_objects as go
import pandas as pd

fieldlist = [1,2,3,4]

# Create an empty figure
fig = go.Figure()

for i in fieldlist:
    df = pd.read_csv(f'fieldsComplete/field_f{i}')
    fig.add_trace(go.Scatter(
        x=df["datetime"],
        y=df["s_a"].to_list(),
        mode='lines',
        name=f"Field {i} in Line",
        visible='legendonly'
    ))
    fig.add_trace(go.Scatter(
        x=df["datetime"],
        y=df["s_b"].to_list(),
        mode='lines',
        name=f"Field {i} between Line",
        visible='legendonly'
    ))
    fig.add_trace(go.Scatter(
        x=df["datetime"],
        y=df["irr"].to_list(),
        mode='lines',
        name=f"Field {i} irrigation",
        visible='legendonly'
    ))
    fig.add_trace(go.Scatter(
        x=df["datetime"],
        y=df["LAI"].to_list(),
        mode='lines',
        name=f"Field {i} LAI",
        visible='legendonly'
    ))

fig.add_trace(go.Scatter(
    x=df["datetime"],
    y=df["RG"],
    mode='markers',
    marker=dict(
        size=5,
    ),
    name=f"rain Station"
))

fig.add_trace(go.Scatter(
    x=df["datetime"],
    y=df["rain"],
    mode='markers',
    marker=dict(
        size=5,
    ),
    name=f"rain Meteo"
))

fig.show()